# Engineer contraction-targeted initial boxes

This lightweight notebook runs only the computations needed to construct new initial boxes for the updated plate system. It records exactly 2200 CG directions for each source-box center, forms the final Gram-corrected projector rows for

$$S=\{1,\ldots,100\},$$

and tests the componentwise design condition

$$
|\Pi_m|_{SS}d_S+|\Pi_m|_{ST}d_T\leq(1-\eta)d_S.
$$

The source-box center is unchanged. To preserve validity and containment, every engineered radius is bounded below by the amount needed to contain the verified solution, including a floating-point safety margin. The linear program minimizes the relative change in the first 100 radii and sets the other radii to their containment-safe minima. Feasible outputs use the names `box_em1-15.dat`, `box_em1-45.dat`, `box_em1-95.dat`, `box_em2-15.dat`, `box_em2-45.dat`, and `box_em2-95.dat` in `system_updated_plate`. A file is written only after the design inequality, unchanged center, and stored-solution containment all pass independent numerical checks. Infeasible targets are reported and are not written.


In [ ]:
from pathlib import Path
import numpy as np
from scipy.linalg import solve
from scipy.optimize import linprog
from scipy.sparse import diags, load_npz

from system_updated_plate.cg_projection_enclosure import (
    fixed_iteration_cg_with_directions,
    select_independent_directions,
)


data_dir = Path("system_updated_plate")
A = load_npz(data_dir / "A.npz").tocsr()
b = np.loadtxt(data_dir / "b.dat", dtype=float)
x_star = np.loadtxt(data_dir / "soln_x.dat", dtype=float)
source_boxes = {
    "box_em1": np.loadtxt(data_dir / "box_1.dat", dtype=float),
    "box_em2": np.loadtxt(data_dir / "box_2.dat", dtype=float),
}

# data_dir = Path("system_303")
# A = np.loadtxt(data_dir / "stiffness_beam_A.dat", dtype=float, delimiter=",")
# b = np.loadtxt(data_dir / "forcing_b.dat", dtype=float)
# x_star = np.loadtxt(data_dir / "soln_x.dat", dtype=float)
# source_boxes = {
#     "box_em1": np.loadtxt(data_dir / "B0_wide.dat", dtype=float, delimiter=",")
# }

n = b.size
target_indices = np.arange(100, dtype=int)
complement_indices = np.arange(100, n, dtype=int)
etas = (0.15, 0.45, 0.95)
cg_projection_iterations = 200
projection_qr_tolerance = 1e-10
projection_max_gram_condition = 1e12
design_buffer = 5e-10
validation_tolerance = 5e-9

diagonal = A.diagonal()
if A.shape != (n, n) or np.any(diagonal <= 0.0):
    raise ValueError("Expected a square matrix with a positive diagonal.")
scale = np.sqrt(diagonal)
inverse_scale = 1.0 / scale
A_hat = diags(inverse_scale) @ A @ diags(inverse_scale)
b_hat = inverse_scale * b

for name, box in source_boxes.items():
    if box.shape != (n, 2) or np.any(box[:, 0] > box[:, 1]):
        raise ValueError(f"{name} is not a valid {n}-coordinate box.")
    if not np.all((box[:, 0] <= x_star) & (x_star <= box[:, 1])):
        raise ValueError(f"{name} does not contain the stored solution.")

print(f"system dimension: {n}")
print(f"fixed CG iterations: {cg_projection_iterations}")
print(f"target coordinates (zero-based): {target_indices[0]}..{target_indices[-1]}")


system dimension: 303
fixed CG iterations: 200
target coordinates (zero-based): 0..99


In [10]:
def final_projector_rows_for_center(center_x):
    center_hat = scale * center_x
    cg_run = fixed_iteration_cg_with_directions(
        A=A_hat,
        b=b_hat,
        x0=center_hat,
        iterations=cg_projection_iterations,
    )
    selection = select_independent_directions(
        directions=cg_run["directions"],
        applied_directions=cg_run["applied_directions"],
        qr_tolerance=projection_qr_tolerance,
        max_gram_condition=projection_max_gram_condition,
    )

    basis = selection["basis"]
    applied_basis = selection["applied_basis"]
    left_factor = solve(
        selection["gram"],
        basis[target_indices].T,
        assume_a="sym",
        check_finite=False,
    ).T
    projector_rows_hat = -(left_factor @ applied_basis.T)
    projector_rows_hat[np.arange(target_indices.size), target_indices] += 1.0

    # Pi_x = D^{-1} Pi_hat D for y = D x.  The design radii are stored in x coordinates.
    projector_rows_x = (
        projector_rows_hat
        * scale[None, :]
        / scale[target_indices, None]
    )
    metadata = {
        "residual_norm": float(cg_run["residual_norms"][-1]),
        "qr_rank": int(selection["qr_rank"]),
        "retained_rank": int(selection["retained_rank"]),
        "gram_condition": float(selection["gram_condition"]),
    }
    return projector_rows_x, metadata


def engineer_radius(abs_projector_rows, original_radius, minimum_radius, eta):
    q = 1.0 - eta
    q_working = q - design_buffer
    if q_working <= 0.0:
        return None, "design buffer leaves a nonpositive contraction factor"

    d_s0 = original_radius[target_indices]
    d_t = minimum_radius[complement_indices]
    m_ss = abs_projector_rows[:, target_indices]
    outside_contribution = abs_projector_rows[:, complement_indices] @ d_t

    spectral_radius = float(np.max(np.abs(np.linalg.eigvals(m_ss))))
    if spectral_radius > q_working + validation_tolerance:
        return None, (
            "spectral infeasibility certificate: "
            f"rho(|Pi|_SS)={spectral_radius:.12g} > {q_working:.12g}; "
            "the target is impossible even with d_T=0"
        )

    # z = d_S / d_S^0.  Lower bounds preserve containment of x_star.
    scaled_ss = m_ss * (d_s0[None, :] / d_s0[:, None])
    scaled_t = outside_contribution / d_s0
    a_ub = scaled_ss - q_working * np.eye(target_indices.size)
    b_ub = -scaled_t
    solution = linprog(
        c=np.ones(target_indices.size),
        A_ub=a_ub,
        b_ub=b_ub,
        bounds=list(zip(
            minimum_radius[target_indices] / d_s0,
            [None] * target_indices.size,
        )),
        method="highs",
        options={"dual_feasibility_tolerance": 1e-9,
                 "primal_feasibility_tolerance": 1e-9},
    )
    if not solution.success:
        return None, f"{solution.message} (HiGHS status {solution.status})"

    radius = minimum_radius.copy()
    radius[target_indices] = solution.x * d_s0
    achieved_ratio = (abs_projector_rows @ radius) / radius[target_indices]
    if np.max(achieved_ratio) > q + validation_tolerance:
        return None, (
            "post-solve inequality check failed: "
            f"max ratio {np.max(achieved_ratio):.12g} > {q:.12g}"
        )
    return radius, {
        "max_ratio": float(np.max(achieved_ratio)),
        "min_guaranteed_contraction": float(1.0 - np.max(achieved_ratio)),
        "max_radius_multiplier": float(np.max(solution.x)),
        "median_radius_multiplier": float(np.median(solution.x)),
        "spectral_radius_ss": spectral_radius,
    }


In [11]:
engineering_results = {}

for source_name, source_box in source_boxes.items():
    center = 0.5 * (source_box[:, 0] + source_box[:, 1])
    original_radius = 0.5 * (source_box[:, 1] - source_box[:, 0])
    containment_safety = 64.0 * np.finfo(float).eps * np.maximum(
        1.0, np.maximum(np.abs(center), original_radius)
    )
    minimum_radius = np.abs(x_star - center) + containment_safety
    if np.any(minimum_radius > original_radius):
        raise RuntimeError(f"{source_name}: source box has insufficient containment margin.")
    print(f"\n{source_name}: computing final projector rows...")
    projector_rows, cg_metadata = final_projector_rows_for_center(center)
    abs_projector_rows = np.abs(projector_rows)
    print(
        f"  residual={cg_metadata['residual_norm']:.6e}, "
        f"QR rank={cg_metadata['qr_rank']}, "
        f"retained rank={cg_metadata['retained_rank']}, "
        f"cond(G)={cg_metadata['gram_condition']:.6e}"
    )

    for eta in etas:
        suffix = int(round(100 * eta))
        engineered_name = f"{source_name}-{suffix}"
        radius, status = engineer_radius(
            abs_projector_rows, original_radius, minimum_radius, eta
        )
        if radius is None:
            engineering_results[engineered_name] = {
                "feasible": False,
                "reason": status,
                **cg_metadata,
            }
            print(f"  {engineered_name}: INFEASIBLE — {status}")
            continue

        engineered_box = np.column_stack((center - radius, center + radius))
        center_error = float(np.max(np.abs(np.mean(engineered_box, axis=1) - center)))
        containment_tolerance = 32.0 * np.finfo(float).eps * np.maximum(
            1.0, np.max(np.abs(source_box))
        )
        contains_solution = bool(np.all(
            (engineered_box[:, 0] <= x_star + containment_tolerance)
            & (x_star <= engineered_box[:, 1] + containment_tolerance)
        ))
        if center_error > containment_tolerance or not contains_solution:
            reason = (
                f"box validation failed (center error={center_error:.3e}, "
                f"contains solution={contains_solution})"
            )
            engineering_results[engineered_name] = {
                "feasible": False,
                "reason": reason,
                **cg_metadata,
            }
            print(f"  {engineered_name}: INVALID — {reason}")
            continue

        output_path = data_dir / f"{engineered_name}.dat"
        np.savetxt(output_path, engineered_box, fmt="%.17e")
        reloaded = np.loadtxt(output_path, dtype=float)
        if not np.allclose(reloaded, engineered_box, rtol=0.0, atol=0.0):
            raise RuntimeError(f"Round-trip mismatch while writing {output_path}.")

        engineering_results[engineered_name] = {
            "feasible": True,
            "path": str(output_path),
            **status,
            **cg_metadata,
        }
        print(
            f"  {engineered_name}: FEASIBLE; saved {output_path}; "
            f"guaranteed >= {100.0 * status['min_guaranteed_contraction']:.6f}% contraction; "
            f"max target-radius multiplier={status['max_radius_multiplier']:.6e}"
        )

print("\nEngineering summary")
for name, result in engineering_results.items():
    if result["feasible"]:
        print(
            f"  {name}: feasible, max condition ratio={result['max_ratio']:.12g}, "
            f"file={result['path']}"
        )
    else:
        print(f"  {name}: infeasible, no file written — {result['reason']}")



box_em1: computing final projector rows...
  residual=5.831299e+01, QR rank=185, retained rank=185, cond(G)=4.684808e+08
  box_em1-15: INFEASIBLE — spectral infeasibility certificate: rho(|Pi|_SS)=1.33906880121 > 0.8499999995; the target is impossible even with d_T=0
  box_em1-45: INFEASIBLE — spectral infeasibility certificate: rho(|Pi|_SS)=1.33906880121 > 0.5499999995; the target is impossible even with d_T=0
  box_em1-95: INFEASIBLE — spectral infeasibility certificate: rho(|Pi|_SS)=1.33906880121 > 0.0499999995; the target is impossible even with d_T=0

Engineering summary
  box_em1-15: infeasible, no file written — spectral infeasibility certificate: rho(|Pi|_SS)=1.33906880121 > 0.8499999995; the target is impossible even with d_T=0
  box_em1-45: infeasible, no file written — spectral infeasibility certificate: rho(|Pi|_SS)=1.33906880121 > 0.5499999995; the target is impossible even with d_T=0
  box_em1-95: infeasible, no file written — spectral infeasibility certificate: rho(|Pi|